# Figure 3b — Experimental vs Numerical (LBM) Velocity Fields
**Main validation figure** — 3×3 grid: columns = `30°` / `40°` / `Irregular`, rows = `Exp` / `LBM` / `Error`.

Conventions follow `Combined_exp_sim_interstices_comparison_FINAL.ipynb`:
- **Exp** CSVs: `x, z, Vel_u, Vel_w` (x–z plane velocity)
- **LBM/sim** CSVs: `Points_0`(x), `Points_2`(z), `av_u_0`(u), `av_u_2`(w), with a z-velocity offset
- Scattered points → regular grid via `scipy.griddata`; `jet` contourf + white normalized streamlines
- **Error row**: Exp & LBM interpolated onto a shared grid per column, then subtracted

👉 **Edit the 6 file paths in the `CASES` dict, then Run All.**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from scipy.interpolate import griddata

## 1. User input — the 6 CSV files (Exp + LBM for each of the 3 cases)

In [ ]:
CASES = {
    "30°": {
        "exp": "/path/to/30deg_experiment.csv",
        "lbm": "/path/to/30deg_lbm.csv",
    },
    "40°": {
        "exp": "/path/to/40deg_experiment.csv",
        "lbm": "/path/to/40deg_lbm.csv",
    },
    "Irregular": {
        "exp": "/path/to/irregular_experiment.csv",
        "lbm": "/path/to/irregular_lbm.csv",
    },
}

OUTPUT_PATH = "Figure_3b_Exp_vs_LBM_validation.png"

## 2. Settings

In [ ]:
N_GRID            = 400        # interpolation grid resolution per axis
N_LEVELS          = 50         # contour levels
Z_VELOCITY_OFFSET = 0.05       # [m/s] added to LBM w-velocity (matches notebook); set 0 to disable
STREAM_DENSITY    = 1.2
ERROR_MODE        = "signed"   # "signed" -> LBM - Exp (diverging) ; "abs" -> |LBM - Exp|
TRIM_PERCENTILE   = 99.0       # clip colorbar limits to this percentile to avoid outlier washout

# --- Error expressed as a PERCENTAGE ---------------------------------
# error% = (LBM - Exp) / V_ref * 100
#   "scale" -> V_ref = global characteristic velocity (vmax across all panels) [robust]
#   "local" -> V_ref = local Exp magnitude              [classic relative error, but
#                                                        noisy / blows up where Exp ~ 0]
ERROR_PCT_REF   = "scale"
LOCAL_REF_FLOOR = 0.02         # [m/s] floor on local Exp value when ERROR_PCT_REF="local"

# Column names
EXP_COLS = dict(x="x", z="z", u="Vel_u", w="Vel_w")
LBM_COLS = dict(x="Points_0", z="Points_2", u="av_u_0", w="av_u_2")

## 3. Helper functions

In [ ]:
def load_field(path, cols, apply_offset=False):
    """Read a CSV and return scattered (x, z, u, w, |V|) arrays."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]          # strip stray header spaces
    x = df[cols["x"]].to_numpy(float)
    z = df[cols["z"]].to_numpy(float)
    u = df[cols["u"]].to_numpy(float)
    w = df[cols["w"]].to_numpy(float)
    if apply_offset:
        w = w + Z_VELOCITY_OFFSET
    mag = np.sqrt(u ** 2 + w ** 2)
    ok = np.isfinite(x) & np.isfinite(z) & np.isfinite(u) & np.isfinite(w)
    return x[ok], z[ok], u[ok], w[ok], mag[ok]


def interp_to_grid(x, z, values, Xi, Zi):
    """Interpolate scattered `values` onto the regular grid (Xi, Zi)."""
    pts = np.column_stack((x, z))
    return griddata(pts, values, (Xi, Zi), method="linear", fill_value=np.nan)


def make_grid(x_lo, x_hi, z_lo, z_hi, n=N_GRID):
    xi = np.linspace(x_lo, x_hi, n)
    zi = np.linspace(z_lo, z_hi, n)
    return np.meshgrid(xi, zi)


def normalised_vectors(U, W):
    """Unit-length vectors for clean streamlines (NaN -> 0 so streamplot is happy)."""
    U = np.nan_to_num(U)
    W = np.nan_to_num(W)
    mag = np.sqrt(U ** 2 + W ** 2)
    Un = np.zeros_like(U)
    Wn = np.zeros_like(W)
    m = mag > 1e-9
    Un[m] = U[m] / mag[m]
    Wn[m] = W[m] / mag[m]
    return Un, Wn

## 4. Load + interpolate every case onto a common grid

In [ ]:
def build_case(paths):
    ex, ez, eu, ew, emag = load_field(paths["exp"], EXP_COLS, apply_offset=False)
    lx, lz, lu, lw, lmag = load_field(paths["lbm"], LBM_COLS, apply_offset=Z_VELOCITY_OFFSET != 0)

    # common (overlapping) physical window so Exp & LBM are sampled identically
    x_lo = max(ex.min(), lx.min());  x_hi = min(ex.max(), lx.max())
    z_lo = max(ez.min(), lz.min());  z_hi = min(ez.max(), lz.max())
    Xi, Zi = make_grid(x_lo, x_hi, z_lo, z_hi)

    field = {
        "Xi": Xi, "Zi": Zi,
        "exp_mag": interp_to_grid(ex, ez, emag, Xi, Zi),
        "exp_U":   interp_to_grid(ex, ez, eu,   Xi, Zi),
        "exp_W":   interp_to_grid(ex, ez, ew,   Xi, Zi),
        "lbm_mag": interp_to_grid(lx, lz, lmag, Xi, Zi),
        "lbm_U":   interp_to_grid(lx, lz, lu,   Xi, Zi),
        "lbm_W":   interp_to_grid(lx, lz, lw,   Xi, Zi),
    }

    # raw signed error [m/s] (only where both datasets are defined);
    # converted to a percentage further below once the velocity scale is known
    field["diff"] = field["lbm_mag"] - field["exp_mag"]
    valid = np.isfinite(field["diff"])
    if valid.sum() > 1:
        a = field["exp_mag"][valid]; b = field["lbm_mag"][valid]
        field["corr"] = float(np.corrcoef(a, b)[0, 1])
    else:
        field["corr"] = np.nan
    return field


fields = {name: build_case(p) for name, p in CASES.items()}

# shared colour scale for the Exp & LBM rows (jet)
all_mag = np.concatenate([
    np.concatenate([f["exp_mag"][np.isfinite(f["exp_mag"])],
                    f["lbm_mag"][np.isfinite(f["lbm_mag"])]])
    for f in fields.values()
])
vmin = float(np.nanmin(all_mag))
vmax = float(np.nanpercentile(all_mag, TRIM_PERCENTILE))
mag_levels = np.linspace(vmin, vmax, N_LEVELS)

# ---- convert the error to a PERCENTAGE -------------------------------
# error% = (LBM - Exp) / V_ref * 100  ; V_ref = global vmax ("scale") or local Exp ("local")
for f in fields.values():
    if ERROR_PCT_REF == "local":
        ref = np.maximum(np.abs(f["exp_mag"]), LOCAL_REF_FLOOR)
    else:
        ref = vmax
    pct = f["diff"] / ref * 100.0                          # signed error [%]
    f["error"] = np.abs(pct) if ERROR_MODE == "abs" else pct
    v = np.isfinite(pct)
    f["rmse"]   = float(np.sqrt(np.nanmean(pct[v] ** 2)))  # all metrics now in [%]
    f["mae"]    = float(np.nanmean(np.abs(pct[v])))
    f["maxerr"] = float(np.nanmax(np.abs(pct[v])))

print(f"Error reference: {ERROR_PCT_REF}"
      + (f"  (V_ref = {vmax:.4f} m/s)" if ERROR_PCT_REF == 'scale' else ''))

# symmetric / positive scale for the error row (units = %)
all_err = np.concatenate([f["error"][np.isfinite(f["error"])] for f in fields.values()])
if ERROR_MODE == "abs":
    err_vmin, err_vmax = 0.0, float(np.nanpercentile(all_err, TRIM_PERCENTILE))
    err_cmap, err_norm = "inferno", None
else:
    e = float(np.nanpercentile(np.abs(all_err), TRIM_PERCENTILE))
    err_vmin, err_vmax = -e, e
    err_cmap, err_norm = "RdBu_r", TwoSlopeNorm(vmin=-e, vcenter=0.0, vmax=e)
err_levels = np.linspace(err_vmin, err_vmax, N_LEVELS)
print("Loaded cases:", list(fields))

## 5. Plot the 3×3 grid

In [ ]:
col_names = list(CASES.keys())
row_names = ["Exp", "LBM", "Error"]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))

vel_handle = None
err_handle = None

for j, cname in enumerate(col_names):
    f = fields[cname]
    Xi, Zi = f["Xi"], f["Zi"]

    panels = [
        ("Exp",   f["exp_mag"], f["exp_U"], f["exp_W"]),
        ("LBM",   f["lbm_mag"], f["lbm_U"], f["lbm_W"]),
        ("Error", f["error"],   None,       None),
    ]

    for i, (rname, mag, U, W) in enumerate(panels):
        ax = axes[i, j]

        if rname == "Error":
            cs = ax.contourf(Xi, Zi, mag, levels=err_levels, cmap=err_cmap,
                             norm=err_norm, extend="both")
            err_handle = cs
            txt = (f"RMSE={f['rmse']:.1f}%\n"
                   f"MAE={f['mae']:.1f}%\n"
                   f"max={f['maxerr']:.1f}%\n"
                   f"r={f['corr']:.3f}")
            ax.text(0.97, 0.04, txt, transform=ax.transAxes, ha="right", va="bottom",
                    fontsize=9, family="monospace",
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.85))
        else:
            cs = ax.contourf(Xi, Zi, mag, levels=mag_levels, cmap="jet",
                             vmin=vmin, vmax=vmax, extend="both")
            vel_handle = cs
            Un, Wn = normalised_vectors(U, W)
            ax.streamplot(Xi, Zi, Un, Wn, density=STREAM_DENSITY,
                          color="white", linewidth=1.0, arrowsize=1.0, arrowstyle="fancy")

        ax.set_xlim(Xi.min(), Xi.max())
        ax.set_ylim(Zi.min(), Zi.max())
        ax.tick_params(labelsize=10)

        if i == 0:
            ax.set_title(cname, fontsize=18, fontweight="bold", pad=10)
        if j == 0:
            ax.set_ylabel(rname, fontsize=18, fontweight="bold", labelpad=12)
        else:
            ax.tick_params(labelleft=False)
        if i == len(panels) - 1:
            ax.set_xlabel("X [m]", fontsize=12)
        else:
            ax.tick_params(labelbottom=False)

fig.suptitle("Figure 3b — Experimental vs Numerical (LBM) Velocity Fields",
             fontsize=22, fontweight="bold", y=0.98)

fig.subplots_adjust(left=0.07, right=0.88, top=0.92, bottom=0.07, hspace=0.10, wspace=0.06)

cax1 = fig.add_axes([0.90, 0.42, 0.018, 0.46])
cb1 = fig.colorbar(vel_handle, cax=cax1)
cb1.set_label("Velocity magnitude |V| [m/s]", rotation=270, labelpad=22,
              fontsize=13, fontweight="bold")

cax2 = fig.add_axes([0.90, 0.07, 0.018, 0.24])
cb2 = fig.colorbar(err_handle, cax=cax2)
err_lbl = "|LBM − Exp|  [%]" if ERROR_MODE == "abs" else "Error  (LBM − Exp)  [%]"
cb2.set_label(err_lbl, rotation=270, labelpad=22, fontsize=13, fontweight="bold")
cb2.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))

fig.savefig(OUTPUT_PATH, dpi=300, bbox_inches="tight")
print(f"Saved figure -> {os.path.abspath(OUTPUT_PATH)}")
plt.show()

## 6. Validation metrics summary

In [ ]:
print("Validation metrics — velocity-magnitude error as % of V_ref:")
print(f"{'case':<12}{'RMSE%':>10}{'MAE%':>10}{'max%':>10}{'corr':>8}")
for name, f in fields.items():
    print(f"{name:<12}{f['rmse']:>9.2f}%{f['mae']:>9.2f}%{f['maxerr']:>9.2f}%{f['corr']:>8.3f}")